# FactSet SPAR Engine to OneLake

Runs one multi-unit SPAR calculation covering every **strategy x SPAR component** pair,
lands the raw STACH response as an audit artifact, and merges a tidy result set into
`Tables/factset/spar_statistics`.

Requires a Fabric **Python** notebook (not PySpark) with `hbcm_datahub` attached as the
default lakehouse. Credentials and workspace configuration come from the `HBCM_Config`
variable library; nothing is hardcoded.

In [ ]:
# deltalake, duckdb and polars ship with the Python notebook runtime; only the FactSet
# packages need installing. restartPython clears state, so it runs before anything else.
%pip install --quiet fds.sdk.SPAREngine fds.protobuf.stach.extensions

notebookutils.session.restartPython()

## Parameters

Pipeline, scheduled and Job Scheduler API runs override these; interactive runs use the
defaults. Dates accept absolute `YYYYMMDD` or relative forms (`-3Y`, `-1Q`, `0`).

In [ ]:
START_DATE = "-3Y"
END_DATE = "0"
FREQUENCY = "Monthly"
CURRENCY_ISO = "USD"
USE_EACH_PORTFOLIO_INCEPTION = False  # True starts each strategy at its own inception
AS_OF_DATE = ""                       # "YYYY-MM-DD"; empty uses today (UTC)
WRITE_RAW = True
DRY_RUN = False                       # True builds and validates the request, submits nothing

## Configuration

`COMPONENTS` holds the saved SPAR component ids; the same set runs for every strategy, so
the request is their cross product. `peer_universe_id` is optional - leave it `None` where
peer-relative statistics are not wanted.

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import pyarrow as pa
from urllib3 import Retry

import fds.sdk.SPAREngine
from fds.sdk.SPAREngine.api import spar_calculations_api
from fds.sdk.SPAREngine.models import (
    CalculationMeta,
    SPARCalculationParameters,
    SPARCalculationParametersRoot,
    SPARDateParameters,
    SPARIdentifier,
)
from fds.protobuf.stach.extensions.StachExtensionFactory import StachExtensionFactory
from fds.protobuf.stach.extensions.StachVersion import StachVersion
from deltalake import DeltaTable, write_deltalake

cfg = notebookutils.variableLibrary.getLibrary("HBCM_Config")

COMPONENTS = {
    "<FILL component key>": "<FILL SPAR component id>",
}

STRATEGIES = {
    "LC": {
        "account_id": "<FILL account id>",
        "account_prefix": "<FILL account prefix>",
        "benchmark_id": "<FILL benchmark id>",
        "benchmark_prefix": "<FILL benchmark prefix>",
        "peer_universe_id": None,
    },
    "LCS": {
        "account_id": "<FILL account id>",
        "account_prefix": "<FILL account prefix>",
        "benchmark_id": "<FILL benchmark id>",
        "benchmark_prefix": "<FILL benchmark prefix>",
        "peer_universe_id": None,
    },
    "SMID": {
        "account_id": "<FILL account id>",
        "account_prefix": "<FILL account prefix>",
        "benchmark_id": "<FILL benchmark id>",
        "benchmark_prefix": "<FILL benchmark prefix>",
        "peer_universe_id": None,
    },
}

ACCOUNT_RETURN_TYPE = "Net"       # portfolios post-fee
BENCHMARK_RETURN_TYPE = "Total"

DEADLINE_SECONDS = 15             # soft deadline for a synchronous 200; multi-unit always polls
POLL_INTERVAL_SECONDS = 3
TIMEOUT_SECONDS = 900

LAKEHOUSE = "/lakehouse/default"
TABLE_PATH = f"{LAKEHOUSE}/Tables/factset/spar_statistics"
RAW_DIR = f"{LAKEHOUSE}/Files/raw/spar"
MERGE_KEYS = ["as_of_date", "strategy", "component", "table_index", "row_label", "statistic"]

as_of_date = AS_OF_DATE or datetime.now(timezone.utc).strftime("%Y-%m-%d")
extracted_at_utc = datetime.now(timezone.utc)

placeholders = [
    f"COMPONENTS[{key}]"
    for key, value in COMPONENTS.items()
    if str(key).startswith("<FILL") or str(value).startswith("<FILL")
] + [
    f"STRATEGIES[{name}].{field}"
    for name, spec in STRATEGIES.items()
    for field, value in spec.items()
    if isinstance(value, str) and value.startswith("<FILL")
]
if placeholders:
    raise ValueError("Unfilled configuration placeholders: " + ", ".join(placeholders))

print(
    f"{len(STRATEGIES)} strategies x {len(COMPONENTS)} components = "
    f"{len(STRATEGIES) * len(COMPONENTS)} calculation units, as of {as_of_date}"
)

## Helpers

In [ ]:
def spar_api_client():
    """SPAR ApiClient authenticated with the HBCM_Config API key."""
    configuration = fds.sdk.SPAREngine.Configuration(
        username=cfg.FACTSET_USERNAME,
        password=cfg.FACTSET_API_KEY,
    )
    # 4xx other than 429 is never retried; urllib3 honours Retry-After on 429 by default.
    configuration.retries = Retry(
        total=3,
        status_forcelist=[500, 502, 503, 504],
        backoff_factor=2,
        allowed_methods=frozenset(["GET", "POST"]),
    )
    return fds.sdk.SPAREngine.ApiClient(configuration)


def build_unit(strategy, component_id):
    """One calculation unit: a strategy's account and benchmark against one SPAR component."""
    spec = STRATEGIES[strategy]
    optional = {"universeid": spec["peer_universe_id"]} if spec["peer_universe_id"] else {}
    return SPARCalculationParameters(
        componentid=component_id,
        accounts=[
            SPARIdentifier(
                id=spec["account_id"],
                returntype=ACCOUNT_RETURN_TYPE,
                prefix=spec["account_prefix"],
            )
        ],
        benchmark=SPARIdentifier(
            id=spec["benchmark_id"],
            returntype=BENCHMARK_RETURN_TYPE,
            prefix=spec["benchmark_prefix"],
        ),
        dates=SPARDateParameters(
            startdate=START_DATE,
            enddate=END_DATE,
            frequency=FREQUENCY,
            useeachportfolioinception=USE_EACH_PORTFOLIO_INCEPTION,
        ),
        currencyisocode=CURRENCY_ISO,
        **optional,
    )


def submit_and_collect(api_client, params_root):
    """Submit one multi-unit calculation; return (calculation_id, {unit_id: (status, result)})."""
    calc_api = spar_calculations_api.SPARCalculationsApi(api_client)
    wrapper = calc_api.post_and_calculate(
        x_fact_set_api_long_running_deadline=DEADLINE_SECONDS,
        spar_calculation_parameters_root=params_root,
    )
    code = wrapper.get_status_code()
    if code == 200:
        status_root = wrapper.get_response_200()
    elif code == 201:
        status_root = wrapper.get_response_201()
    elif code == 202:
        status_root = wrapper.get_response_202()
        calculation_id = status_root.data.calculationid
        expires_at = time.time() + TIMEOUT_SECONDS
        while True:
            if time.time() > expires_at:
                calc_api.cancel_calculation_by_id(id=calculation_id)
                raise TimeoutError(
                    f"SPAR calculation {calculation_id} exceeded {TIMEOUT_SECONDS}s and was cancelled"
                )
            poll = calc_api.get_calculation_status_by_id(id=calculation_id)
            poll_code = poll.get_status_code()
            if poll_code == 200:
                status_root = poll.get_response_200()
                break
            if poll_code != 202:
                raise RuntimeError(f"Unexpected status {poll_code} polling {calculation_id}")
            time.sleep(POLL_INTERVAL_SECONDS)
    else:
        raise RuntimeError(f"Unexpected status {code} from post_and_calculate")

    calculation_id = status_root.data.calculationid
    collected = {}
    for unit_id, unit_status in status_root.data.units.items():
        result = None
        if unit_status.status == "Success":
            result = calc_api.get_calculation_unit_result_by_id(id=calculation_id, unit_id=unit_id)
        collected[unit_id] = (unit_status.status, result)
    return calculation_id, collected


def stach_to_frames(result):
    """STACH JsonStach payload to one DataFrame per returned table."""
    extension = StachExtensionFactory.get_stach_extension(StachVersion.V2)
    tables = extension.convert(json.dumps(result.to_dict()))
    return [pd.DataFrame(table.data, columns=table.columns) for table in tables]


def resolve_duplicate_labels(labels):
    """Duplicate display labels are valid SPAR output; disambiguate deterministically."""
    seen, resolved = {}, []
    for label in labels:
        seen[label] = seen.get(label, 0) + 1
        resolved.append(label if seen[label] == 1 else f"{label}#{seen[label]}")
    return resolved


TIDY_COLUMNS = [
    "strategy", "component", "table_index", "row_label", "statistic",
    "statistic_label", "label_column", "value_raw",
]


def tidy_tables(unit_id, frames):
    """Unpivot each returned table to one row per (row_label, statistic). Empty tables persist."""
    strategy, component = unit_id.split("__", 1)
    tidied = []
    for table_index, frame in enumerate(frames):
        if frame.empty or frame.shape[1] < 2:
            tidied.append(pd.DataFrame(columns=TIDY_COLUMNS))
            continue
        original = list(frame.columns)
        resolved = resolve_duplicate_labels(original)
        renamed = frame.copy()
        renamed.columns = resolved
        label_column = resolved[0]
        long = renamed.melt(id_vars=[label_column], var_name="statistic", value_name="value_raw")
        long = long.rename(columns={label_column: "row_label"})
        long["statistic_label"] = long["statistic"].map(dict(zip(resolved, original)))
        long["strategy"] = strategy
        long["component"] = component
        long["table_index"] = table_index
        long["label_column"] = original[0]
        tidied.append(long[TIDY_COLUMNS])
    return tidied

## Fetch

A multi-unit calculation always runs asynchronously, so the helper polls regardless of the
deadline. Unit ids are `{strategy}__{component}` and become the join keys on the way out.

In [ ]:
units = {
    f"{strategy}__{component_key}": build_unit(strategy, component_id)
    for strategy in STRATEGIES
    for component_key, component_id in COMPONENTS.items()
}

params_root = SPARCalculationParametersRoot(
    data=units,
    meta=CalculationMeta(
        contentorganization="SimplifiedRow",
        stach_content_organization="SimplifiedRow",
        contenttype="Json",
        format="JsonStach",
    ),
)

calculation_id = None
collected = {}

if DRY_RUN:
    print(f"DRY_RUN: built {len(units)} units, submitted nothing -> {sorted(units)}")
else:
    with spar_api_client() as api_client:
        calculation_id, collected = submit_and_collect(api_client, params_root)
    succeeded = sum(1 for status, _ in collected.values() if status == "Success")
    print(f"calculation {calculation_id}: {succeeded}/{len(units)} units succeeded")

## Decode and reconcile

Each unit is reconciled against its own source tables before anything is combined: a tidy
table must hold `rows x (columns - 1)` records. Values are kept both as text and as a
coerced numeric; non-numeric cells leave `value_num` null rather than zero.

In [ ]:
decoded = {}
reconciliation = []

for unit_id, (status, result) in collected.items():
    if result is None:
        reconciliation.append({
            "unit_id": unit_id, "status": status, "table_index": None,
            "source_rows": 0, "source_columns": 0, "expected_rows": 0, "tidy_rows": 0,
        })
        continue
    frames = stach_to_frames(result)
    tidied = tidy_tables(unit_id, frames)
    decoded[unit_id] = tidied
    for table_index, (source, tidy) in enumerate(zip(frames, tidied)):
        expected = 0 if source.empty or source.shape[1] < 2 else len(source) * (source.shape[1] - 1)
        reconciliation.append({
            "unit_id": unit_id, "status": status, "table_index": table_index,
            "source_rows": len(source), "source_columns": source.shape[1],
            "expected_rows": expected, "tidy_rows": len(tidy),
        })

reconciliation = pd.DataFrame(reconciliation)
row_mismatches = (
    reconciliation[reconciliation["tidy_rows"] != reconciliation["expected_rows"]]
    if not reconciliation.empty else reconciliation
)

frames_to_combine = [frame for tidied in decoded.values() for frame in tidied]
statistics = (
    pd.concat(frames_to_combine, ignore_index=True)
    if frames_to_combine else pd.DataFrame(columns=TIDY_COLUMNS)
)
statistics["as_of_date"] = as_of_date
statistics["value_num"] = pd.to_numeric(statistics["value_raw"], errors="coerce")
statistics["value_text"] = statistics["value_raw"].map(lambda v: None if pd.isna(v) else str(v))
statistics = statistics.drop(columns=["value_raw"])
statistics["calculation_id"] = calculation_id
statistics["extracted_at_utc"] = extracted_at_utc

display(reconciliation)

## Validate

Decoding validity and publication policy are separate questions: a structurally valid run
that returns no rows is reported, not published, and not treated as a decode failure.

In [ ]:
failures = []

if not DRY_RUN:
    missing_units = sorted(set(units) - set(collected))
    if missing_units:
        failures.append(f"units absent from the response: {missing_units}")

    failed_units = sorted(u for u, (status, _) in collected.items() if status != "Success")
    if failed_units:
        failures.append(f"units did not succeed: {failed_units}")

    if not row_mismatches.empty:
        failures.append("row reconciliation mismatch:\n" + row_mismatches.to_string(index=False))

    duplicate_keys = int(statistics.duplicated(subset=MERGE_KEYS).sum())
    if duplicate_keys:
        failures.append(f"{duplicate_keys} rows collide on the merge key {MERGE_KEYS}")

    unlabelled = int(statistics["row_label"].isna().sum())
    if unlabelled:
        failures.append(f"{unlabelled} rows carry a null row_label and cannot be keyed")

if failures:
    raise ValueError("Validation failed:\n- " + "\n- ".join(failures))

non_numeric = int(statistics["value_num"].isna().sum()) if not statistics.empty else 0
print(
    f"{len(statistics)} rows across {statistics['strategy'].nunique() if not statistics.empty else 0} "
    f"strategies; {non_numeric} non-numeric values retained as text"
)
display(statistics.head(20))

## Publish

Raw responses land as one JSON file per run - flat under `Files/raw/spar/`, dated in the
filename - so a reshape never costs another API call. The curated table is merged on
`MERGE_KEYS`, so a re-run restates its vintage instead of duplicating it.

In [ ]:
raw_path = None
published_rows = 0

if DRY_RUN:
    print("DRY_RUN: nothing published")
elif statistics.empty:
    print("no rows decoded; nothing published")
else:
    if WRITE_RAW:
        Path(RAW_DIR).mkdir(parents=True, exist_ok=True)
        raw_path = f"{RAW_DIR}/spar_{extracted_at_utc:%Y%m%dT%H%M%SZ}.json"
        Path(raw_path).write_text(json.dumps({
            "calculation_id": calculation_id,
            "as_of_date": as_of_date,
            "parameters": {
                "startdate": START_DATE, "enddate": END_DATE, "frequency": FREQUENCY,
                "currencyisocode": CURRENCY_ISO,
                "useeachportfolioinception": USE_EACH_PORTFOLIO_INCEPTION,
                "account_returntype": ACCOUNT_RETURN_TYPE,
                "benchmark_returntype": BENCHMARK_RETURN_TYPE,
            },
            "units": {
                unit_id: (result.to_dict() if result is not None else None)
                for unit_id, (_, result) in collected.items()
            },
        }, default=str))

    arrow_table = pa.Table.from_pandas(statistics, preserve_index=False)
    if Path(TABLE_PATH, "_delta_log").exists():
        predicate = " AND ".join(f"target.{key} = source.{key}" for key in MERGE_KEYS)
        (
            DeltaTable(TABLE_PATH)
            .merge(
                source=arrow_table,
                predicate=predicate,
                source_alias="source",
                target_alias="target",
            )
            .when_matched_update_all()
            .when_not_matched_insert_all()
            .execute()
        )
    else:
        Path(TABLE_PATH).parent.mkdir(parents=True, exist_ok=True)
        write_deltalake(TABLE_PATH, arrow_table, mode="overwrite")
    published_rows = len(statistics)

summary = {
    "as_of_date": as_of_date,
    "calculation_id": calculation_id,
    "units_requested": len(units),
    "units_succeeded": sum(1 for status, _ in collected.values() if status == "Success"),
    "rows_published": published_rows,
    "table_path": TABLE_PATH,
    "raw_path": raw_path,
    "extracted_at_utc": extracted_at_utc.isoformat(),
}
print(summary)

# exit() raises internally, so it must stay outside any try/except to take effect.
notebookutils.notebook.exit(json.dumps(summary))